# 03 – Feature Engineering & Data Preprocessing

Raw data is rarely ready for a model.  Feature engineering can make or break performance.

Topics covered:
1. Handling missing values
2. Encoding categorical variables (Label, One-Hot, Ordinal)
3. Feature scaling (MinMax, Standard, Robust)
4. Feature creation (polynomial, log, interaction)
5. Handling outliers
6. Pipelines for reproducible preprocessing

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

%matplotlib inline
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. Handling Missing Values

In [ ]:
# Simulate a dataset with missing values
df = pd.DataFrame({
    'age':    [25, np.nan, 35, 28, np.nan, 45, 33, 29],
    'salary': [50000, 62000, np.nan, 58000, 71000, np.nan, 54000, 67000],
    'dept':   ['HR', 'IT', 'IT', np.nan, 'Finance', 'HR', np.nan, 'Finance'],
    'target': [0, 1, 1, 0, 1, 0, 1, 0]
})

print(df)
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
# Strategies
num_cols = ['age', 'salary']

# Mean imputation
imp_mean = SimpleImputer(strategy='mean')
df_mean  = df.copy()
df_mean[num_cols] = imp_mean.fit_transform(df[num_cols])

# Median imputation
imp_med = SimpleImputer(strategy='median')
df_med  = df.copy()
df_med[num_cols] = imp_med.fit_transform(df[num_cols])

# KNN imputation
imp_knn = KNNImputer(n_neighbors=2)
df_knn  = df.copy()
df_knn[num_cols] = imp_knn.fit_transform(df[num_cols])

# Categorical – most frequent
imp_cat = SimpleImputer(strategy='most_frequent')
df_mean['dept'] = imp_cat.fit_transform(df[['dept']]).ravel()

print('After mean imputation:\n', df_mean)

## 2. Encoding Categorical Variables

In [ ]:
sample = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size':  ['small', 'medium', 'large', 'medium', 'small'],
    'score': [1, 2, 3, 4, 5]
})

# Label Encoding (ordinal-like)
le = LabelEncoder()
sample['color_le'] = le.fit_transform(sample['color'])

# One-Hot Encoding
ohe = pd.get_dummies(sample['color'], prefix='color')

# Ordinal Encoding (for size: small < medium < large)
oe = OrdinalEncoder(categories=[['small', 'medium', 'large']])
sample['size_ord'] = oe.fit_transform(sample[['size']])

print(sample)
print('\nOne-Hot:\n', ohe)

## 3. Feature Scaling

In [ ]:
data = np.array([[100, 0.01], [200, 0.02], [300, 0.03],
                 [400, 0.04], [10000, 0.1]])   # last row = outlier in col 0

scalers = {
    'Original':    data,
    'StandardScaler': StandardScaler().fit_transform(data),
    'MinMaxScaler':   MinMaxScaler().fit_transform(data),
    'RobustScaler':   RobustScaler().fit_transform(data)
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, arr) in zip(axes, scalers.items()):
    ax.scatter(arr[:, 0], arr[:, 1])
    ax.set_title(name)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

## 4. Feature Creation

In [ ]:
X_base = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])

# Polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_base)
print('Original columns:', X_base.shape[1])
print('After degree-2 poly:', X_poly.shape[1])
print('Feature names:', poly.get_feature_names_out(['x1', 'x2']))

In [ ]:
# Log transform for skewed distributions
income = np.array([30000, 35000, 40000, 55000, 90000, 120000, 500000])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(income, bins=7, color='steelblue', edgecolor='k'); axes[0].set_title('Original')
axes[1].hist(np.log1p(income), bins=7, color='coral', edgecolor='k'); axes[1].set_title('Log Transform')
plt.tight_layout(); plt.show()

## 5. Full Pipeline with ColumnTransformer

In [ ]:
from sklearn.datasets import fetch_openml

# Titanic
titanic = fetch_openml('titanic', version=1, as_frame=True)
df_t = titanic.frame[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'survived']].dropna()
X_t = df_t.drop(columns='survived')
y_t = df_t['survived'].astype(int)

num_features = ['age', 'fare', 'sibsp', 'parch']
cat_features = ['sex', 'pclass']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf',          LogisticRegression(max_iter=1000))
])

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_t, y_t, test_size=0.2, random_state=42)
full_pipe.fit(X_tr_t, y_tr_t)

print('Test Accuracy:', accuracy_score(y_te_t, full_pipe.predict(X_te_t)):.3f)
cv = cross_val_score(full_pipe, X_t, y_t, cv=5)
print(f'CV: {cv.mean():.3f} ± {cv.std():.3f}')

## 6. Key Takeaways

| Technique | When to use |
|-----------|-------------|
| Mean/Median impute | Numeric; median is more robust to outliers |
| KNN impute | When missingness has a pattern |
| One-Hot encode | Nominal categories (no order) |
| Ordinal encode | Ordered categories |
| StandardScaler | Most ML algorithms (SVM, kNN, neural nets) |
| RobustScaler | Data with outliers |
| Polynomial features | Capture non-linear relationships in linear models |
| Log transform | Right-skewed numeric features |
| Pipeline | Prevents data leakage; easy deployment |

**Next:** `04_Model_Evaluation.ipynb`